# Lab 23: FedSpeak 2.0 — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 63 min Core + 60 min AI + 55 min Extension

---

**Learning Objectives:**
- Find three planted errors in an analyst's NLP pipeline for FOMC minutes, one in each of its three steps, and say what each does to the numbers after it
- Read a regular expression piece by piece, and say what it keeps as a token and what it drops
- Score each meeting with word lists in the style of Loughran-McDonald, and say why a general-purpose list misreads central-bank text
- Build a TF-IDF matrix whose vocabulary leaves out the terms every meeting uses
- Write your own prompt for a text cleaner, and check it against this lab's token counts and against sentences it has never seen

**Dataset:** FOMC meeting minutes: 243 meetings, February 2000 to July 2026, from the HuggingFace dataset `vtasca/fomc-statements-minutes`, saved on 4 October 2026 so everyone works on the same text. It is the file ECON 3916's Lab 23 uses.

**Format:** Part 1 runs an analyst's pipeline with three errors in it, one in each of its three steps, and asks what each one is. Part 2 fixes them: Fixes 1 and 2 are given, and you write Fix 3. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Parts 1 and 2), your Fix 3, the AI-Assisted Expansion (your prompts, the checking log and two answers), and `src/fomc_sentiment.py`, a finished module you save and upload.

**Time:** Core (63 min) + AI-Assisted Expansion (required, 60 min) + Extension (optional: Part 3, 25 min, and the Challenge, 30 min)

With setup, the README and the GitHub submission, plan on about 2 hours 15 minutes, most of it after class. Part 3 and the Challenge add about 55 minutes.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only in the AI-Assisted Expansion, where you write the prompt yourself.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Part 2, Fix 3:** a TF-IDF matrix with the settings Part 2 names, about five lines in an empty cell, in the shape of Part 1's Step 3.
> - **Part 3 and the Challenge (optional):** sentence-transformer embeddings, clustering and a prediction test, from outlines.
>
> **You write, in text cells:**
> - three answers in Part 1 and two in Part 2;
> - **the AI-Assisted Expansion (required):** your own prompt, from scratch, asking AI for a text cleaner, `clean_tokens()`, that gives the same tokens as Fix 1's `fed_tokens()`; a revised prompt for round 2 if round 1 shows a `NO` or an error (and round 3, in its own cells, if you need it); a short checking log; and two answers. Your prompts, the log and the answers are what is graded, not the AI's code, which you paste rather than type.
>
> **You run and read:** everything else. That is loading the minutes, the repair of mis-read characters, the analyst's three steps, Fix 1 (`fed_tokens()`, the cleaner the AI section asks you to describe in words), Fix 2 (the same scoring with Loughran-McDonald-style lists), the checkpoint after Fix 3, and Part 4's finished module, `fomc_sentiment.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints. The check cells in the AI section are tools: run them and read what they print; their code is not part of the lab.

## Setup

Colab comes with NLTK (the Natural Language Toolkit) and scikit-learn, so nothing needs installing.
NLTK keeps its word lists as separate data files: `nltk.download("stopwords")` fetches its lists of
stop words, the common words such as "the" and "and" that most text analysis drops, and
`quiet=True` hides the download messages.

`Counter` (from Python's `collections`) counts how often each item occurs in whatever you give it.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries this lab uses, and NLTK's lists of stop words
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download("stopwords", quiet=True)
Path("src").mkdir(exist_ok=True)     # Part 4 writes its module into this folder
print("Setup complete.")

## Part 1: DIAGNOSE — Find 3 Errors in This NLP Pipeline (33 min)

The Federal Open Market Committee (FOMC) sets U.S. interest rates. A few weeks after each meeting it
publishes **minutes**: a long account of what the staff reported and what the members said. Markets
read the wording closely, and an analyst wants a sentiment score for every meeting and a TF-IDF
matrix to cluster and predict with. Her pipeline has three steps, one cell each, and each of the
three cells has one error. Run them, read what they print, and answer the three questions after
Step 3.

**Loading the minutes.** The minutes were saved as one compressed CSV file, one row per meeting,
in date order: the `date` and the `text` of the minutes. The live dataset gains a row whenever new
minutes come out, so the saved copy keeps your numbers equal to the expected outputs.
- `pd.read_csv` reads the `.gz` file directly: it sees the ending and unzips the file as it reads.
- `try` runs the lines under it; if they raise an error (no internet, say), `except` runs instead
  and reads a copy from a `data` folder beside the notebook. That copy exists only in a downloaded
  copy of the course repository, not in Colab: in Colab, this cell needs the internet.
- `pd.to_datetime` turns the date strings into dates, and `.dt.year` takes the year of each.

In [ ]:
# GIVEN — run as-is
# Load the saved FOMC minutes, one row per meeting
DATA_URL = ("https://raw.githubusercontent.com/RyanPiao/econ5200-student-notebooks/main/"
            "5200/data/lab-ch23-fomc-minutes-2026-10.csv.gz")
LOCAL_COPY = "data/lab-ch23-fomc-minutes-2026-10.csv.gz"

try:
    fomc_df = pd.read_csv(DATA_URL)
except Exception:                       # works only in a downloaded copy of the course repository
    fomc_df = pd.read_csv(LOCAL_COPY)

fomc_df["date"] = pd.to_datetime(fomc_df["date"])
fomc_df["year"] = fomc_df["date"].dt.year
print(f"{len(fomc_df)} meetings, {fomc_df['date'].min().date()} to {fomc_df['date'].max().date()}")

**Expected output:** `243 meetings, 2000-02-02 to 2026-07-29`.

**Repairing mis-read characters.** When the dataset was built, some characters were garbled: an em
dash "—" stored as three bytes of UTF-8 and read back one byte at a time comes out as `â€”`, and
Windows quotation marks come out as invisible control characters. Such garbled text is called
**mojibake**. The next cell finds each garbled sequence, turns it back into its bytes
(`.encode("latin-1")`) and reads the bytes the right way (`.decode("utf-8")`, or `"cp1252"` for the
Windows ones). `re.sub(pattern, function, text)` calls the function on each match `m`, and
`m.group()` is the matched text; `.str.count(pattern)` counts the matches in each set of minutes.

In [ ]:
# GIVEN — run as-is
# Repair the characters that were mis-read when the dataset was built
MISREAD_UTF8 = "â\x80.|Å.|ï¿½"     # a UTF-8 character read one byte at a time
MISREAD_1252 = "[\x80-\x9f]"        # Windows punctuation read as control characters
MISREAD = MISREAD_UTF8 + "|" + MISREAD_1252


def repair(text):
    # each mis-read sequence back to its bytes, then the bytes read the right way
    text = re.sub(MISREAD_UTF8, lambda m: m.group().encode("latin-1").decode("utf-8"), text)
    return re.sub(MISREAD_1252, lambda m: m.group().encode("latin-1").decode("cp1252"), text)


print(f"mis-read sequences before: {fomc_df['text'].str.count(MISREAD).sum():,}")
fomc_df["text"] = fomc_df["text"].apply(repair)
print(f"mis-read sequences after:  {fomc_df['text'].str.count(MISREAD).sum():,}")

**Expected output:** 1,509 mis-read sequences before and 0 after. Fix 1, which keeps only
a to z, would give the same tokens either way, but a cleaner that treats `â` as a letter would read
`inflationâ€”as` as one word.

**Step 1, reading the analyst's code.** `.split()` cuts a string at every space, tab or line
break. `[t for t in ... if ...]` (a list comprehension) keeps the tokens that pass the test after
`if`, and a `set` makes `t not in stop_words` a fast lookup. `.apply(analyst_tokens)` runs the
function on each set of minutes, so `raw_tokens` holds one list of tokens per meeting.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 1: a list of tokens for every set of minutes
stop_words = set(stopwords.words("english"))


def analyst_tokens(text):
    # one document's tokens: lower case, split, stop words dropped
    return [t for t in text.lower().split() if t not in stop_words]


fomc_df["raw_tokens"] = fomc_df["text"].apply(analyst_tokens)
print(f"Stop words in the list: {len(stop_words)}")
print("The first 20 tokens of the first minutes:")
print(fomc_df["raw_tokens"].iloc[0][:20])

**Step 1, continued** counts every token of every meeting with `Counter`. The sum of the counts
is the number of tokens, and `len(raw_counts)` the number of different tokens, the **vocabulary**.
`.most_common()` lists tokens from the most frequent down, and `:,` puts commas in a number.

In [ ]:
# GIVEN — run as-is
# Step 1, continued: how many tokens, how many different ones, and those that contain "inflation"
raw_counts = Counter(t for tokens in fomc_df["raw_tokens"] for t in tokens)
print(f"tokens in the first minutes: {len(fomc_df['raw_tokens'].iloc[0]):,}")
print(f"tokens in all {len(fomc_df)} minutes: {sum(raw_counts.values()):,}")
print(f"different tokens (the vocabulary): {len(raw_counts):,}")
print("the eight commonest tokens that contain 'inflation':")
print([(t, n) for t, n in raw_counts.most_common() if "inflation" in t][:8])

**Expected output:** `Stop words in the list: 198`, the first 20 tokens from `'minutes'` to
`'system'`, then 5,163 tokens in the first minutes, 1,103,660 in all of them, a vocabulary
of 19,954, and eight tokens that contain "inflation", from `('inflation', 10463)` down.
NLTK has changed its stop-word list over the years: an older copy prints 179, without words such
as "i'm" and "i'd". The minutes use five of those, so with the older list Step 1's counts come out
up to 5 tokens higher. Fix 1 splits at apostrophes, so from Fix 1 on nothing changes.

**Step 2: sentiment from word lists.** The next cell holds two functions that Part 2 uses again.
`sentiment` scores one meeting: (positive count − negative count) / tokens. Chapter 23
(Section 23.3) divides by positive + negative + 1 instead; the sign is the same either way.
`sum(t in neg_words for t in tokens)` counts the tokens in the list, because True counts as 1.
`top_hits` gives the `n` commonest tokens, over all meetings, that are in a word list.

In [ ]:
# GIVEN — run as-is
# Step 2: score one meeting with two word lists, and count a list's commonest words
def sentiment(tokens, neg_words, pos_words):
    # (positive - negative) / tokens, with the two counts
    neg = sum(t in neg_words for t in tokens)
    pos = sum(t in pos_words for t in tokens)
    return {"net": (pos - neg) / len(tokens), "neg": neg, "pos": pos}


def top_hits(token_lists, words, n=10):
    # the n commonest tokens, over all meetings, that are in the word list
    return Counter(t for tokens in token_lists for t in tokens if t in words).most_common(n)

**Reading the analyst's code.** `gi_negative` and `gi_positive` are short lists in the style of the
Harvard General Inquirer (GI) dictionary. `pd.DataFrame` of a list of dictionaries, one per meeting,
makes one row per meeting and one column per key. `.idxmin()` gives the row label of the smallest
value, and `.loc[row, "date"]` that row's date.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 2: net sentiment for every meeting, from Harvard GI-style word lists
gi_negative = {
    "abandon", "adverse", "against", "bad", "blame", "capital", "concern", "cost", "costly",
    "crisis", "danger", "debt", "decline", "deficit", "difficult", "expense", "fail", "failure",
    "fear", "liability", "limit", "limitation", "loss", "negative", "obligation", "penalty",
    "problem", "risk", "tax", "threat", "trouble", "uncertain", "unemployment", "volatile",
    "weak", "worse",
}
gi_positive = {
    "achieve", "advantage", "benefit", "confidence", "gain", "good", "growth", "improve",
    "increase", "opportunity", "positive", "profit", "progress", "strong", "success", "value",
}

gi_scores = pd.DataFrame([sentiment(tokens, gi_negative, gi_positive)
                          for tokens in fomc_df["raw_tokens"]])
print(f"mean net sentiment: {gi_scores['net'].mean():.6f}")
print(f"most negative meeting: {fomc_df.loc[gi_scores['net'].idxmin(), 'date'].date()}")
print("the ten commonest 'negative' words:")
print(top_hits(fomc_df["raw_tokens"], gi_negative))

**Expected output:** a mean net sentiment of 0.001176, the most negative meeting on 2009-06-03, and
ten words from `('unemployment', 2177)` to `('tax', 329)`.

**Step 3: the TF-IDF matrix.** `TfidfVectorizer` builds one row per meeting and one column per term,
from strings, so `.str.join(" ")` first glues each meeting's tokens back into one string.
`tokenizer=str.split` cuts each string at the spaces, so its terms are exactly the analyst's
tokens. `token_pattern=None` turns off scikit-learn's own rule for finding words, and
`lowercase=False` leaves the case alone. `fit_transform` learns the vocabulary and builds the matrix.

**The weight in each cell** is tf × idf. tf is how often the term occurs in that meeting's minutes.
idf is scikit-learn's ln((1 + n) / (1 + df)) + 1, where n is the number of meetings (243) and df the
number that use the term: a term used in all 243 gets the smallest idf, ln(244/244) + 1 = 1 (not
0), and a term used in 5 gets ln(244/6) + 1 = 4.71. Last, each meeting's row is scaled to length 1,
so long and short minutes can be compared.

`min_df` and `max_df` take a whole number (a count of documents) or a decimal between 0 and 1 (a
share of documents): a term is kept if it is used in at least `min_df` and at most `max_df` of the
minutes. `max_features` keeps at most that many terms, the ones used most often, and
`ngram_range=(1, 1)` keeps single words only; `(1, 2)` also keeps pairs of neighbouring words.

The matrix is **sparse**: it stores only the entries that are not zero, and `.nnz` counts them, so
`1 - nnz / (rows × columns)` is the share of zeros. `top_terms`, in the cell after it, prints the
terms with the highest average TF-IDF and how many meetings use each; Part 2 uses it again.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# Step 3: a TF-IDF matrix of the analyst's tokens
analyst_tfidf = TfidfVectorizer(
    tokenizer=str.split, token_pattern=None, lowercase=False,   # the terms are the tokens
    min_df=1,              # a count: used in at least 1 meeting
    max_df=1.0,            # a share: used in at most 100% of them
    max_features=10000,    # the 10,000 terms used most often
    ngram_range=(1, 1),    # single words
)
analyst_matrix = analyst_tfidf.fit_transform(fomc_df["raw_tokens"].str.join(" "))
rows, cols = analyst_matrix.shape
print(f"TF-IDF matrix: {rows} meetings x {cols} terms, {1 - analyst_matrix.nnz / (rows * cols):.2%} zeros")

`top_terms` averages each column over the meetings: `matrix.mean(axis=0)` gives the averages and
`np.asarray(...).ravel()` turns them into a flat array. `(matrix > 0).sum(axis=0)` counts, for each
term, the meetings that use it. `.argsort()` gives the positions that would sort the averages from
smallest to largest, `[::-1]` turns that round, and `[:n]` keeps the first `n`. `:25s` pads a term
to 25 characters so the numbers line up.

In [ ]:
# GIVEN — run as-is
# Step 3, continued: the 15 terms with the highest average TF-IDF, and how many meetings use each
def top_terms(matrix, vectorizer, n=15):
    # the n terms with the highest average TF-IDF over all meetings
    terms = vectorizer.get_feature_names_out()
    average = np.asarray(matrix.mean(axis=0)).ravel()
    used_in = np.asarray((matrix > 0).sum(axis=0)).ravel()
    for i in average.argsort()[::-1][:n]:
        print(f"  {terms[i]:25s} {average[i]:.4f}   in {used_in[i]} of {matrix.shape[0]} meetings")


top_terms(analyst_matrix, analyst_tfidf)

**Expected output:** `TF-IDF matrix: 243 meetings x 10000 terms, 84.08% zeros`, then fifteen
terms from `federal` (0.1780) to `remained`, fourteen of them used in all 243 meetings.

### YOUR DIAGNOSIS

1. **Step 1.** What is wrong with the analyst's tokens? Quote two pieces of output that show it,
   and say what it does to the counts that Steps 2 and 3 make from them.
2. **Step 2.** Which of the ten commonest "negative" words would you not call negative in FOMC
   minutes, and why? What does that do to the mean net sentiment, and to the meeting the analyst
   would call the most negative?
3. **Step 3.** What is wrong with the fifteen top terms, and which of the settings caused it? Say
   what a term that every set of minutes uses can tell you about one meeting.

*Your diagnosis here:*

---

## Part 2: FIX — Correct the Pipeline (25 min)

Each fix replaces one step and keeps the rest: Fix 1 replaces the tokenizer, Fix 2 the word lists,
and Fix 3 the TF-IDF settings. Fixes 1 and 2 are given. You write Fix 3.

### Fix 1: tokens from a regular expression (given)

A **regular expression** (regex) is a small pattern that describes text. `re.findall(pattern, text)`
gives back, as a list of strings in order, every piece of `text` the pattern matches; whatever lies
between the matches is dropped. Here is the pattern Fix 1 uses, `\d+(?:\.\d+)?%?|[a-z]+`, piece by
piece:

| piece | matches |
|---|---|
| `\d+` | one or more digits: `2`, `2000` |
| `(?:\.\d+)?` | then, optionally, a full stop and more digits: the `.5` of `2.5`. `\.` is a full stop itself (a bare `.` would match any character), `(?: )` groups the two without capturing, and `?` makes the group optional |
| `%?` | then, optionally, a percent sign |
| `[a-z]+` | one or more letters from a to z |

The bar `|` between `%?` and `[a-z]+` means *or*: at each place in the text, the pattern tries a
number first and a run of letters second. So a token is a number, with a decimal part and a percent sign if it has them, or a run of letters.
Every other character separates tokens: a space, a line break, a comma, a hyphen, an apostrophe, a
full stop that is not between digits. The `r` before the string ("raw") keeps Python from reading
`\d` itself, so the backslash reaches `re`. The group has to be `(?: )`: with plain brackets,
`re.findall` would give back only what the brackets matched, `.5`, not the whole number.

The letters are a to z only, so `fed_tokens` lowercases first. After the pattern, it drops the
same stop words as Step 1.

In [ ]:
# GIVEN — run as-is
# Fix 1: a token is a number (with its decimal part and % sign) or a run of letters
TOKEN = r"\d+(?:\.\d+)?%?|[a-z]+"


def fed_tokens(text):
    # one document's tokens: lower case, the pattern's matches, stop words dropped
    return [t for t in re.findall(TOKEN, text.lower()) if t not in stop_words]


print(fed_tokens("The U.S. 10-year yield rose to 4.5% in mid-May; the Committee's view held."))

**Expected output:** `['u', '10', 'year', 'yield', 'rose', '4.5%', 'mid', 'may', 'committee', 'view', 'held']`. The hyphens split `10-year` and `mid-May`, `4.5%` survives whole,
and `U.S.` leaves only `u`: the full stops separate `u` from `s`, and `s` is a stop word, as `t`
is (from "doesn't"). The apostrophe in `Committee's` does the same.

**Fix 1, continued** cleans every set of minutes into the column `tokens` and prints the numbers
the AI-Assisted Expansion checks against: the same three counts as Step 1, continued, and how often
six tokens occur. Each of the six depends on one choice: `inflation` on punctuation, `committee`
on apostrophes ("Committee's"), `term` on hyphens ("longer-term"), `percent` and `2.5` on numbers,
and `u` on full stops ("U.S."). `WATCHED` lists the six. The cell takes a few seconds.

In [ ]:
# GIVEN — run as-is
# Fix 1, continued: clean every set of minutes, and count what that made
fomc_df["tokens"] = fomc_df["text"].apply(fed_tokens)
counts = Counter(t for tokens in fomc_df["tokens"] for t in tokens)
print(f"tokens in the first minutes: {len(fomc_df['tokens'].iloc[0]):,}")
print(f"tokens in all {len(fomc_df)} minutes: {sum(counts.values()):,}")
print(f"different tokens (the vocabulary): {len(counts):,}")

WATCHED = ["inflation", "committee", "term", "percent", "2.5", "u"]
for word in WATCHED:
    print(f'  count of "{word}": {counts[word]:,}')

**Expected output:** 5,186 tokens in the first minutes, 1,119,019 in all of them, a vocabulary
of 9,703, and counts of `inflation` 12,214, `committee` 12,805, `term` 4,708, `percent` 4,599, `2.5` 33 and `u` 2,239. These nine numbers are the AI-Assisted Expansion's target.

### Fix 2: Loughran-McDonald-style word lists

The **Loughran-McDonald (LM) dictionary** is the standard for financial text. It was built from
companies' annual reports (10-K filings), after its authors showed that a general-purpose list
misreads financial text. The next cell holds two short lists in its style, a few dozen words each,
picked for this lab (the master dictionary is at the link in the cell). Fix 1 does not lemmatize,
so a list needs each form of a word: `decline`, `declined` and `declining` are three entries.

In [ ]:
# GIVEN — run as-is
# Fix 2: two short word lists in the style of Loughran-McDonald
# The LM master dictionary: https://sraf.nd.edu/loughranmcdonald-master-dictionary/
lm_negative = {
    "adverse", "adversely", "against", "concern", "concerned", "concerns", "decline", "declined",
    "declining", "decrease", "decreased", "deficit", "deteriorate", "deteriorated",
    "deteriorating", "difficult", "difficulty", "downturn", "fail", "failure", "falling", "loss",
    "losses", "negative", "negatively", "recession", "recessionary", "risk", "risks", "risky",
    "severe", "severely", "slowdown", "sluggish", "stress", "stressed", "threat", "threaten",
    "troubled", "uncertain", "uncertainty", "unfavorable", "volatile", "volatility",
    "vulnerability", "vulnerable", "weak", "weaken", "weakened", "weakness", "worse", "worsen",
    "worsened", "worsening", "worst",
}
lm_positive = {
    "achieve", "achieved", "achievement", "benefit", "beneficial", "confidence", "confident",
    "favorable", "gain", "gained", "gains", "good", "growth", "improve", "improved",
    "improvement", "improving", "increase", "increased", "opportunity", "opportunities",
    "optimism", "optimistic", "positive", "positively", "profit", "profitable", "progress",
    "prosper", "prosperity", "rebound", "recover", "recovered", "recovery", "strength",
    "strengthen", "strengthened", "strong", "stronger", "strongest", "success", "successful",
    "upturn", "upward",
}
print(f"words in the lists: {len(lm_negative)} negative, {len(lm_positive)} positive")

**Expected output:** `words in the lists: 55 negative, 44 positive`.

The next cell is Step 2 again, with Fix 1's tokens and the LM lists in place of the analyst's.

In [ ]:
# GIVEN — run as-is
# Fix 2: net sentiment for every meeting, from Fix 1's tokens and the LM lists
lm_scores = pd.DataFrame([sentiment(tokens, lm_negative, lm_positive)
                          for tokens in fomc_df["tokens"]])
print(f"mean net sentiment: {lm_scores['net'].mean():.6f}")
print(f"most negative meeting: {fomc_df.loc[lm_scores['net'].idxmin(), 'date'].date()}")
print("the ten commonest negative words:")
print(top_hits(fomc_df["tokens"], lm_negative))

**Expected output:** a mean net sentiment of 0.004838, the most negative meeting on 2008-12-16, and
ten words from `('risks', 3040)` to `('concern', 449)`.

### Fix 3: TF-IDF settings that drop background terms

**Your task.** In the empty cell below, build a TF-IDF matrix as Step 3 did, but from Fix 1's
tokens, and call the vectorizer `tfidf` and the matrix `tfidf_matrix`. Keep Step 3's first line
(`tokenizer=str.split, token_pattern=None, lowercase=False`) and change the four settings to:
- terms used in at least **5 meetings**;
- terms used in at most **85% of the meetings**;
- at most **5,000 terms**;
- single words **and** pairs of neighbouring words.

Then run the checkpoint cell under it.

In [ ]:
# YOUR TASK — Fix 3: tfidf and tfidf_matrix, from Fix 1's tokens

In [ ]:
# GIVEN — run as-is, once Fix 3 is written
# Fix 3, checkpoint: the matrix's size, and its 15 top terms
rows, cols = tfidf_matrix.shape
print(f"TF-IDF matrix: {rows} meetings x {cols} terms, {1 - tfidf_matrix.nnz / (rows * cols):.2%} zeros")
top_terms(tfidf_matrix, tfidf)

**Expected output:** `TF-IDF matrix: 243 meetings x 5000 terms, 67.57% zeros`, then fifteen terms
from `mr` (0.1097, in 148 of 243 meetings) to `longer run`. The most widely used of the
fifteen, `2 percent`, is in 177 meetings. Four near misses: `67.54%` zeros means `min_df`
is a share (0.05) where it should be a count, 5 meetings; `81.90%`, with `shall` at the top,
means `max_df` counts 85 meetings rather than 85% of them; `80.79%`, with no pair of words in the
list, means `ngram_range` stayed `(1, 1)`; and `67.99%`, with no `2 percent`, means Step 3's first
line is missing, so scikit-learn's own rule for words dropped every one-character token, `2` among
them.

### YOUR DIAGNOSIS

1. Fix 1 raised the number of tokens in all the minutes, from Step 1's 1,103,660 to 1,119,019,
   and cut the vocabulary by half. Explain both changes from what the pattern does. Then name one
   token Fix 1 makes, other than `u`, that you would not want in a sentiment score or a TF-IDF
   matrix, and say why.
2. Compare Fix 2's ten negative words with Step 2's. Which of the LM words would you still
   question in FOMC minutes? Then read Fix 3's top terms: what do they pick up about the meetings
   that is not monetary policy, and what would you change to get rid of it?

*Your diagnosis here:*

---

## Part 3: EXTEND — Sentence-Transformers Embeddings (optional — 25 min)

This part is optional, and no required part needs it (only the optional Challenge does): to skip
it, go on to Part 4, and leave its cells unrun, so your submitted notebook shows no error.

TF-IDF treats each word on its own: "the economy is weakening" and "growth is slowing" share no
word, so their TF-IDF rows have nothing in common. A **sentence-transformer** reads a whole passage
and gives back a dense vector, here 384 numbers, in which passages that mean similar things lie
close together (Chapter 23, Section 23.5).

**The model.** `all-MiniLM-L6-v2` is a small model (about 90 MB) that runs on Colab's CPU. The
first line of the next cell installs the `sentence-transformers` package at a fixed version, and
`MODEL_REVISION` fixes the model's version on HuggingFace, so everyone downloads the same model.
If Colab asks you to restart the session after the install, restart it and run the notebook again
from the top.

**Reading `embed`.** The model reads at most 256 word pieces of a passage and ignores the rest, and
a set of minutes is about 7,000 words. So `embed` cuts each set of minutes at its blank lines
(`re.split(r"\n\s*\n", text)`: a line break, any white space, another line break) and keeps the
paragraphs of at least 40 words, which leaves out the list of who attended. It encodes the
paragraphs and averages them, one vector per meeting. `model.encode(list_of_texts)` gives one row of
384 numbers per text, and `.mean(axis=0)` averages the rows. The `import` sits inside the function,
so the rest of the lab runs without the package. `np.vstack` stacks the 243 vectors into one array.

In [ ]:
# GIVEN — run as-is (optional)
# Part 3: a function that turns each set of minutes into one 384-number vector
!pip install -q "sentence-transformers==6.1.0"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"     # the model's version on HuggingFace


def embed(texts, min_words=40):
    # one vector per text: the average of its paragraphs' embeddings
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME, revision=MODEL_REVISION)
    vectors = []
    for text in texts:
        paragraphs = [p for p in re.split(r"\n\s*\n", text) if len(p.split()) >= min_words]
        vectors.append(model.encode(paragraphs, batch_size=32).mean(axis=0))
    return np.vstack(vectors)

**Your task (optional).** In the empty cell below:
1. `embeddings = embed(fomc_df["text"])`. It downloads the model once and then takes 2-5 minutes
   on Colab's CPU (**Runtime → Change runtime type → T4 GPU** makes it faster). Print its shape.
2. Reduce Fix 3's `tfidf_matrix` to 50 columns with
   `TruncatedSVD(n_components=50, random_state=42).fit_transform(...)` (the version of PCA that
   works on a sparse matrix), and call the result `tfidf_reduced`.
3. Cluster both into 3 groups with `KMeans(n_clusters=3, n_init=10, random_state=42)`, as in
   Lab 22, and store the labels in `fomc_df["cluster_emb"]` and `fomc_df["cluster_tfidf"]`.
4. For each clustering print its silhouette score (`silhouette_score(X, labels)`), and each
   cluster's size and first and last date.
5. Print `adjusted_rand_score(fomc_df["cluster_emb"], fomc_df["cluster_tfidf"])`: 1 when the two
   clusterings group the meetings the same way, about 0 when they agree no more than chance.

The imports: `from sklearn.decomposition import TruncatedSVD`, `from sklearn.cluster import
KMeans` and `from sklearn.metrics import silhouette_score, adjusted_rand_score`.

In [ ]:
# YOUR TASK — optional: embeddings, two clusterings, and how far they agree

### Questions (optional)

1. Can the two silhouette scores be compared with each other, to say which representation
   clusters the minutes better? Say why or why not.
2. Using the dates and the adjusted Rand index, say what each clustering groups the meetings by.
   Which one would you show a referee who asks why a cluster exists, and why?

*Your answers here:*

1.
2.

---

## Part 4: Module Output — `fomc_sentiment.py` (5 min)

Here is the finished module, reading only: run the two cells and compare the output with the
expected output below. It holds the three fixed steps of Part 2 as functions, so the next project
starts from the fixed pipeline rather than from the analyst's: `preprocess_fomc` is Fix 1,
`compute_lm_sentiment` is Fix 2 for one meeting, and `build_tfidf_matrix` is Fix 3. The word lists
are the ones in Fix 2. The second cell compares the module with Fix 2 and your Fix 3, so write Fix 3
first.

`%%writefile src/fomc_sentiment.py` saves the cell as that file instead of running it. `-> list`
only notes the type a function gives back.

In [ ]:
%%writefile src/fomc_sentiment.py
"""FOMC minutes as data: tokens, LM-style sentiment and a TF-IDF matrix (ECON 5200 Lab 23)."""
import re

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

TOKEN = r"\d+(?:\.\d+)?%?|[a-z]+"       # a number (decimal part and % sign kept) or a run of letters
STOP_WORDS = set(stopwords.words("english"))

LM_NEGATIVE = {
    "adverse", "adversely", "against", "concern", "concerned", "concerns", "decline", "declined",
    "declining", "decrease", "decreased", "deficit", "deteriorate", "deteriorated",
    "deteriorating", "difficult", "difficulty", "downturn", "fail", "failure", "falling", "loss",
    "losses", "negative", "negatively", "recession", "recessionary", "risk", "risks", "risky",
    "severe", "severely", "slowdown", "sluggish", "stress", "stressed", "threat", "threaten",
    "troubled", "uncertain", "uncertainty", "unfavorable", "volatile", "volatility",
    "vulnerability", "vulnerable", "weak", "weaken", "weakened", "weakness", "worse", "worsen",
    "worsened", "worsening", "worst",
}
LM_POSITIVE = {
    "achieve", "achieved", "achievement", "benefit", "beneficial", "confidence", "confident",
    "favorable", "gain", "gained", "gains", "good", "growth", "improve", "improved",
    "improvement", "improving", "increase", "increased", "opportunity", "opportunities",
    "optimism", "optimistic", "positive", "positively", "profit", "profitable", "progress",
    "prosper", "prosperity", "rebound", "recover", "recovered", "recovery", "strength",
    "strengthen", "strengthened", "strong", "stronger", "strongest", "success", "successful",
    "upturn", "upward",
}


def preprocess_fomc(text) -> list:
    """One document's tokens: lower case, the TOKEN pattern's matches, stop words dropped."""
    return [t for t in re.findall(TOKEN, text.lower()) if t not in STOP_WORDS]


def compute_lm_sentiment(tokens) -> dict:
    """Net sentiment of one document's tokens, (positive - negative) / tokens, with the counts."""
    neg = sum(t in LM_NEGATIVE for t in tokens)
    pos = sum(t in LM_POSITIVE for t in tokens)
    return {"net": (pos - neg) / len(tokens), "neg": neg, "pos": pos}


def build_tfidf_matrix(token_lists, min_df=5, max_df=0.85, max_features=5000) -> tuple:
    """A TF-IDF matrix of words and word pairs; gives back (matrix, terms, vectorizer)."""
    vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                                 min_df=min_df, max_df=max_df, max_features=max_features,
                                 ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([" ".join(tokens) for tokens in token_lists])
    return matrix, vectorizer.get_feature_names_out(), vectorizer

In [ ]:
# GIVEN — run as-is
# Import the module, run it on the raw minutes, and compare with Part 2
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import fomc_sentiment
importlib.reload(fomc_sentiment)   # reload, so a re-run of the %%writefile cell takes effect

module_tokens = fomc_df["text"].apply(fomc_sentiment.preprocess_fomc)
module_net = module_tokens.apply(lambda tokens: fomc_sentiment.compute_lm_sentiment(tokens)["net"])
module_matrix, module_terms, _ = fomc_sentiment.build_tfidf_matrix(module_tokens)

print(f"tokens in all minutes: {module_tokens.str.len().sum():,}   (Fix 1: {sum(counts.values()):,})")
print(f"mean net sentiment   : {module_net.mean():.6f}   (Fix 2: {lm_scores['net'].mean():.6f})")
print(f"TF-IDF matrix        : {module_matrix.shape}   (Fix 3: {tfidf_matrix.shape})")
print(f"the module's top term: {module_terms[np.asarray(module_matrix.mean(axis=0)).argmax()]}")

**Expected output:** each line agrees with Part 2: 1,119,019 tokens, a mean net sentiment of
0.004838, a matrix of `(243, 5000)`, and `mr` as the top term. `.str.len()` counts the tokens in
each meeting's list, and `lambda tokens: ...` is a one-line function that `.apply` calls on each
meeting's list. A difference in the third line means your Fix 3 differs from the module: find out
why before you go on.

---

## AI-Assisted Expansion: a text cleaner, your prompt, checked by you

Fix 1's `fed_tokens()` is two lines, and every part of them is a decision: what counts as a token,
what happens to numbers, hyphens, apostrophes and full stops, and which words are dropped. A text
cleaner is code you write again for every new corpus, and quick for AI to write, because regular
expressions are something it does well. It also makes every decision your prompt leaves out, and on
text like the minutes most of those decisions move a count by less than one percent, which is why
the check prints every number. No prompt is given: you write it.

**The target.** A function `clean_tokens(text)` that gives back the same tokens as `fed_tokens()`,
in the same order, for any text. The check runs it on all 243 sets of minutes and compares Fix 1,
continued's nine numbers: **5,186** tokens in the first minutes, **1,119,019** in all of them, a
vocabulary of **9,703**, and the six counts from `inflation` (**12,214**) to `u` (**2,239**).
Then it runs it on nine short sentences that are not in the minutes, each with something the
minutes rarely or never contain, and compares the tokens one by one: a function that fits the
minutes and nothing else fails there.

**Describe, do not paste.** Write the rules in your own words. Do not paste `fed_tokens`, `TOKEN`
or any other cell of this notebook into the chat: a prompt that hands the AI the answer tests
nothing, and your prompt is what is graded.

**What a good prompt pins down.** Fix 1 and the text around it hold every answer.
- the function's name, its one argument, and what it gives back (a list of strings, in the order
  they occur in the text);
- case: lower case or not;
- what counts as a token, and what separates two tokens: spaces, line breaks, tabs, punctuation;
- numbers: kept or dropped; a decimal such as 2.5; a percent sign; the dollar sign and comma in
  `$2,000`;
- letters: a to z only, or accented letters too, such as the ó in "López" (the minutes name a few
  staff members that way);
- hyphens ("longer-term"), apostrophes ("Committee's") and full stops inside abbreviations
  ("U.S.");
- stop words: which list, from which library, and whether anything else is dropped;
- what to leave out: any step your rules do not name (stemming, lemmatizing, a minimum length),
  example text or a test of its own, and any use of the notebook's data; and a comment on each line.

**What to do**
1. Write your round 1 prompt in the cell below and send it, exactly as written, to Claude
   ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT, in a new chat.
2. Read the reply before you run anything. Paste only the function, with any `import` lines it
   needs, into the round 1 cell, and run the check below it.
3. If a line says `NO`, or the check reports an error, find the choice in the function that caused
   it, state that choice in a round 2 prompt, and paste and check the new function in the round 2
   cells. If you need a third round, use the round 3 cells, so round 2's output stays for your log.
   Stop when every line says `yes` and you can explain each line, or when you decide to stop.
4. If everything says `yes` in round 1, skip rounds 2 and 3. Instead, in Question 1, name a choice
   the AI made that your prompt never stated, and write the line you would add to your prompt to
   state it.
5. Fill in the checking log and answer the two questions. Those and your prompts are graded, not
   the code.

**Reading the check.** It prints one line per number: Fix 1's number, the AI's, `same?`, and the %
apart. Then one line per sentence: what the sentence contains, the number of tokens from
`fed_tokens` and from the AI, the % apart, and whether the tokens are the same; under a sentence
whose tokens differ, it prints the sentence and both lists, and under the numbers, if any says
`NO`, it prints the first meeting whose tokens differ and the tokens around the first difference.
A correct function takes a few seconds on all 243 sets of minutes; if it runs for minutes, look for
something rebuilt for every word, such as `stopwords.words("english")` inside the comprehension.
Your tokens and the AI's come from the
same text in the same session, so a correct function agrees exactly. There is no allowed gap: one
token in a million makes a line say `NO`. The check does not say why a line differs: finding that
out is your job.

Keep the function's name. If the AI also wrote example text or a test, leave that out.

In [ ]:
# GIVEN — run as-is (a tool: its code is not part of the lab; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves the minutes, the stop words, Fix 1's pattern and fed_tokens now, before anything is pasted.
saved_text, saved_stop_words, saved_pattern = fomc_df["text"].copy(), set(stop_words), TOKEN
lab_cleaner = fed_tokens
saved_dates = [str(d.date()) for d in fomc_df["date"]]
saved_tokens = list(fomc_df["tokens"])

# Nine sentences the minutes do not contain, each with one thing a cleaner must decide about
NEW_SENTENCES = [
    ("a percent sign", "Core inflation was 2.5% in May, down from 2.75% in April."),
    ("abbreviations with full stops", "The U.S. economy grew; U.K. output slowed."),
    ("hyphens", "Crypto-asset risks and AI-driven productivity gains were discussed."),
    ("line breaks, tabs, extra spaces", "Rates rose\n\nsharply   in\tthe   second quarter."),
    ("apostrophes", "The Committee's view didn't change."),
    ("capital letters", "INFLATION Remains ELEVATED."),
    ("a dollar sign, a comma and a year", "Purchases of $80,000 million continued into 2026."),
    ("words the minutes never use", "Quantum chatbot tokenization reached the metaverse."),
    ("nothing at all (an empty text)", ""),
]


def corpus_numbers(token_lists):
    """Fix 1, continued's nine numbers, for one list of tokens per meeting."""
    counts = Counter(t for tokens in token_lists for t in tokens)
    numbers = {"tokens in the first minutes": len(token_lists[0]),
               "tokens in all the minutes": sum(counts.values()),
               "vocabulary": len(counts)}
    for word in WATCHED:
        numbers[f'count of "{word}"'] = counts[word]
    return numbers


LAB_NUMBERS = corpus_numbers(list(fomc_df["tokens"]))
LAB_SENTENCES = [lab_cleaner(sentence) for _, sentence in NEW_SENTENCES]


def lab_names_changed():
    """The lab's names that the pasted code redefined, if any."""
    changed = []
    # try runs the lines under it; if they raise an error (say fomc_df is gone), except runs instead
    try:
        if not fomc_df["text"].equals(saved_text):
            changed.append("fomc_df")
    except Exception:
        changed.append("fomc_df")
    for name, saved in [("stop_words", saved_stop_words), ("TOKEN", saved_pattern)]:
        if globals().get(name) != saved:
            changed.append(name)
    if globals().get("fed_tokens") is not lab_cleaner:
        changed.append("fed_tokens")
    return changed


def first_difference(ai_lists):
    """Print the first meeting whose tokens differ, and the tokens around the first difference."""
    for date, mine, ai in zip(saved_dates, saved_tokens, ai_lists):
        if mine != ai:
            i = next((k for k, (a, b) in enumerate(zip(mine, ai)) if a != b), min(len(mine), len(ai)))
            print(f"\n  first difference: the minutes of {date}, token {i + 1}")
            print(f"      fed_tokens: {mine[max(i - 2, 0):i + 6]}")
            print(f"      AI        : {ai[max(i - 2, 0):i + 6]}")
            return


def tokens_from(fn, text, where):
    """The AI's tokens for one text as a list of strings, or None after saying what went wrong."""
    try:
        result = fn(text)
    except Exception as err:
        print(f"On {where}, the AI's function stopped with an error. {type(err).__name__}: {err}")
        return None
    if isinstance(result, str):
        print(f"On {where}, clean_tokens gave back one string, not a list of tokens.")
        return None
    if not isinstance(result, (list, tuple)):
        print(f"On {where}, clean_tokens gave back a {type(result).__name__}. "
              "The check needs a list of strings.")
        return None
    if not all(isinstance(t, str) for t in result):
        kinds = sorted({type(t).__name__ for t in result if not isinstance(t, str)})
        print(f"On {where}, clean_tokens gave back a list that holds {', '.join(kinds)}, "
              "not only strings.")
        return None
    return list(result)


def percent_apart(mine, ai):
    """How far the AI's number is from the lab's, in percent, as text."""
    if mine == 0:
        return "0%" if ai == 0 else "n/a"
    pct = 100 * (ai - mine) / mine
    # under 10%: sign and 2 significant digits, so one token in a million prints like +8.9e-05%
    return f"{pct:+.2g}%" if abs(pct) < 10 else f"{pct:+.0f}%"


def check_clean_tokens(fn):
    """Run the AI's clean_tokens on the minutes and on nine new sentences, beside fed_tokens."""
    changed = lab_names_changed()
    if changed:
        print(f"The pasted code redefined {', '.join(changed)}, which the lab uses. Paste only the "
              "function, and rename anything else it defines with that name (a pattern called "
              "TOKEN, say, could be PATTERN); then rerun the notebook from the top.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("clean_tokens should be a function. Paste the function the AI wrote.")
        return
    # co_names: the names a function's code uses
    if fn is lab_cleaner or "fed_tokens" in getattr(getattr(fn, "__code__", None), "co_names", ()):
        print("This clean_tokens is, or calls, the lab's own fed_tokens, so the check would "
              "compare fed_tokens with itself. Paste the function the AI wrote.")
        return

    ai_lists = []
    for text, date in zip(saved_text, saved_dates):
        tokens = tokens_from(fn, text, f"the minutes of {date}")
        if tokens is None:
            return
        ai_lists.append(tokens)
    ai_numbers = corpus_numbers(ai_lists)

    same_count, lines = 0, len(LAB_NUMBERS) + len(NEW_SENTENCES)
    print(f"  {'Fix 1, continued':<34}{'fed_tokens':>11}{'AI':>11}{'same?':>7}{'% apart':>10}")
    for name, mine in LAB_NUMBERS.items():
        ai = ai_numbers[name]
        same = "yes" if ai == mine else "NO"
        same_count += same == "yes"
        print(f"  {name:<34}{mine:>11,}{ai:>11,}{same:>7}{percent_apart(mine, ai):>10}")
    if same_count < len(LAB_NUMBERS):
        first_difference(ai_lists)

    print(f"\n  {'new sentences: the one with':<34}{'fed_tokens':>11}{'AI':>11}{'same tokens?':>14}"
          f"{'% apart':>10}")
    for (label, sentence), mine in zip(NEW_SENTENCES, LAB_SENTENCES):
        ai = tokens_from(fn, sentence, f"the sentence with {label}")
        if ai is None:
            print(f"  {label:<34}{len(mine):>11}{'error':>11}{'NO':>14}")
            continue
        same = "yes" if ai == mine else "NO"
        same_count += same == "yes"
        print(f"  {label:<34}{len(mine):>11}{len(ai):>11}{same:>14}"
              f"{percent_apart(len(mine), len(ai)):>10}")
        if same == "NO":
            print(f"      text      : {sentence!r}")
            print(f"      fed_tokens: {mine}")
            print(f"      AI        : {ai}")

    print(f"\n{same_count} of {lines} lines say yes.")
    print("If a line says NO, or a sentence stopped with an error, find out why before you trust "
          "the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the clean_tokens function from round 1 here, then run the check below.
# Paste only the function and its imports: if the AI also wrote example text or a test, leave it out.

In [ ]:
# CHECK — round 1
# To rerun this check, rerun the round 1 paste cell first: it checks whatever was pasted last
# globals() holds every name defined so far in this notebook
if "clean_tokens" not in globals():
    print("Paste the AI's clean_tokens function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    clean_tokens_r1 = clean_tokens      # kept, so the round 2 check can tell the two apart
    check_clean_tokens(clean_tokens_r1)

**Your prompt, round 2.** Keep what worked in round 1, and state what the AI had to choose on its
own.

*Your prompt here:*

In [ ]:
# Paste the clean_tokens function from round 2 here, then run the check below.
# Paste only the function and its imports: if the AI also wrote example text or a test, leave it out.

In [ ]:
# CHECK — round 2
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "clean_tokens" not in globals() or clean_tokens is globals().get("clean_tokens_r1"):
    print("Paste the AI's round 2 clean_tokens function in the cell above first, "
          "keeping that name, and run that cell.")
else:
    clean_tokens_r2 = clean_tokens      # kept, so the round 3 check can tell the two apart
    check_clean_tokens(clean_tokens_r2)

**Your prompt, round 3** (only if you need it). Write it here; leave it empty if you stopped
earlier.

*Your prompt here:*

In [ ]:
# Paste the clean_tokens function from round 3 here, if there is one, then run the check below.

In [ ]:
# CHECK — round 3
if "clean_tokens" not in globals() or any(
        clean_tokens is globals().get(name) for name in ["clean_tokens_r1", "clean_tokens_r2"]):
    print("No round 3 function pasted. That is fine if you stopped at round 1 or 2.")
else:
    check_clean_tokens(clean_tokens)

### Checking log

This is graded, with your prompts and the two answers below. One row per round: what you checked, how you checked it, and what you found. A row that
names the lines that said `NO`, how far apart they were, and the line of the AI's code responsible
earns credit; "it worked" does not. The last row is where you stopped, and why.

| Round | What I checked | How | What I found |
|---|---|---|---|
| 1 |  |  |  |
| 2 (write "not needed" if round 1 was all yes) |  |  |  |
| 3 (if any) |  |  |  |
| Where I stopped |  |  |  |

### Questions

1. **Critique, in your own words.** Name one choice the AI made that your prompt never stated,
   quote the line where it makes it, and say whether you would keep it and why. (A choice that
   caused a `NO` or an error counts. If everything said `yes` in round 1, name a choice your prompt
   never stated and write the prompt line that would state it; then name an input on which your
   function and `fed_tokens` might still differ, and try it.) Then quote the line of your final
   function that finds the tokens and explain in 2-3 sentences, in your own words, what each part of
   it matches and why it gives Fix 1's numbers.
2. **Judgment: where you stopped.** Which round did you stop at, and why do you, or do you not,
   trust that function on text the check never tried, such as another central bank's minutes or a
   speech with footnotes? Then: when would you ask AI to write a text cleaner, and when would you
   write it yourself?

*Your answers here:*

1.
2.

---

## Challenge: Compare TF-IDF vs Embedding Predictive Power (optional — 30 min)

This take-home needs Part 3's `embeddings` and `tfidf_reduced`. If you skip it, leave its cells
unrun, so your submitted notebook shows no error.

**The target.** The next cell marks each meeting 1 if it falls in a calendar year in which the FOMC
raised its target rate at least once (2000, 2004-2006, 2015-2018, 2022-2023), and 0 otherwise. It
is a rough label: a year counts even when its only rise came in December, as in 2015 and 2016.
`.isin(...)` is True for the years in the set, and `.astype(int)` turns True and False into 1 and 0.

In [ ]:
# GIVEN — run as-is (optional)
# Challenge: 1 for a meeting in a year with at least one rate rise, 0 otherwise
TIGHTENING_YEARS = {2000, 2004, 2005, 2006, 2015, 2016, 2017, 2018, 2022, 2023}
fomc_df["tightening"] = fomc_df["year"].isin(TIGHTENING_YEARS).astype(int)
print(fomc_df["tightening"].value_counts().sort_index().to_string())

**Expected output:** 163 meetings marked 0 and 80 marked 1.

**Your task (optional).** In the empty cell below, and in new cells under it:
1. With `TimeSeriesSplit(n_splits=5)` (from `sklearn.model_selection`), which always trains on
   earlier meetings and tests on later ones, fit `LogisticRegression(max_iter=1000)` on
   `tfidf_reduced`, and separately on `embeddings`, and score each test fold with
   `roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])` (from `sklearn.metrics`).
2. A test fold can hold only 0s (no rate rose in 2007-2014), and the AUC of such a fold cannot
   be computed. Skip those folds and say how many you skipped.
3. Report each representation's mean AUC and its standard deviation across the folds you kept.

Then write one paragraph in the text cell: which representation predicts better, by how much
compared with the spread across folds, and what the minutes of a tightening year might contain
that makes this label easy or hard to predict.

In [ ]:
# YOUR TASK — optional take-home: TF-IDF against embeddings, AUC by time-series fold

*Your answer here:*

---
## Digital Portfolio: your README

Copy the prompt below into Claude or ChatGPT. Replace every `[YOUR VALUE]` with the number from
your own output first, and do not quote a number this notebook did not print. **Do NOT ask the AI
to write Python code — only documentation.**

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Found three planted errors in an NLP pipeline for [YOUR VALUE] sets
  of FOMC minutes: [ONE LINE FOR EACH ERROR]
* Fixed the tokenizer with a regular expression that keeps numbers:
  the vocabulary went from [YOUR VALUE] to [YOUR VALUE] different tokens
* Scored every meeting with Loughran-McDonald-style word lists: mean
  net sentiment [YOUR VALUE], most negative meeting [YOUR VALUE]
* Built a TF-IDF matrix of [YOUR VALUE] meetings by [YOUR VALUE] terms;
  its top terms were [YOUR VALUE]
* Wrote my own prompt asking an AI for a text cleaner, went [YOUR VALUE]
  rounds, and checked it against nine numbers from the minutes and nine
  new sentences: [YOUR RESULT]
* (Only if I did Part 3 or the Challenge) [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: FedSpeak: Diagnosing an NLP Pipeline for FOMC Minutes
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'cutting-edge'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab23-fedspeak`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab23-fedspeak`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab23-fedspeak`, branch `main`,
   commit message `Lab 23 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/fomc_sentiment.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.